<a href="https://colab.research.google.com/github/Musharrafmrm/hybrid-ecommerce-issue-detection/blob/main/notebooks/08_LDA_and_Feature_Preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# =============================================================================
# 08_LDA_and_Feature_Preparation.ipynb
# Hybrid Machine Learning for E-Commerce Review Issue Detection
# =============================================================================

import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords', quiet=True)

print("Loading cleaned dataset...")
df = pd.read_csv('data/cleaned_dataset.csv')
print(f"Loaded {len(df):,} reviews\n")

# Prepare text for LDA
stop_words = stopwords.words('english')
vectorizer = CountVectorizer(max_df=0.95, min_df=2, stop_words=stop_words, max_features=5000)

print("Converting text to features for LDA...")
X = vectorizer.fit_transform(df['cleaned_text'])

# Train LDA model (5 topics = 5 main issue categories)
print("Training LDA for aspect discovery...")
lda = LatentDirichletAllocation(n_components=5, random_state=42, max_iter=10)
lda.fit(X)

print("✅ LDA training completed!\n")

# Show top words per topic
def print_topics(model, feature_names, n_top_words=10):
    for topic_idx, topic in enumerate(model.components_):
        top_words = [feature_names[i] for i in topic.argsort()[:-n_top_words-1:-1]]
        print(f"Topic {topic_idx+1}: {', '.join(top_words)}")

print("Top words per topic:")
print_topics(lda, vectorizer.get_feature_names_out())

# Save LDA model and vectorizer
import joblib
joblib.dump(lda, 'models/lda_model.pkl')
joblib.dump(vectorizer, 'models/vectorizer.pkl')
print("\n✅ LDA model and vectorizer saved in models/ folder")

# Add topic distribution to dataframe (for later hybrid model)
topic_distribution = lda.transform(X)
for i in range(5):
    df[f'topic_{i+1}'] = topic_distribution[:, i]

df.to_csv('data/prepared_dataset.csv', index=False)
print("✅ Prepared dataset with LDA topics saved as 'data/prepared_dataset.csv'")

Loading cleaned dataset...


FileNotFoundError: [Errno 2] No such file or directory: 'data/cleaned_dataset.csv'

In [2]:
# =============================================================================
# 08_LDA_Feature_Preparation.ipynb
# Hybrid Machine Learning for E-Commerce Review Issue Detection
# =============================================================================

import pandas as pd
import re
import os
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
import joblib

nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

print("🔄 Checking for cleaned dataset...\n")

# Try to load cleaned dataset, if not found → load raw and clean it
if os.path.exists('data/cleaned_dataset.csv'):
    df = pd.read_csv('data/cleaned_dataset.csv')
    print("✅ Loaded cleaned dataset")
else:
    print("Cleaned file not found. Loading raw dataset and cleaning...")
    df = pd.read_csv('data/Reviews.csv')
    df = df[['Score', 'Summary', 'Text']].copy()

    def clean_text(text):
        if not isinstance(text, str):
            return ""
        text = text.lower()
        text = re.sub(r'[^a-zA-Z\s]', '', text)
        text = re.sub(r'\s+', ' ', text).strip()
        return text

    df['cleaned_text'] = df['Text'].apply(clean_text)
    df = df[df['cleaned_text'].str.len() > 15]
    df.to_csv('data/cleaned_dataset.csv', index=False)
    print("✅ Cleaned and saved new dataset")

print(f"Total reviews: {len(df):,}\n")

# Prepare text for LDA
stop_words = stopwords.words('english')
vectorizer = CountVectorizer(max_df=0.95, min_df=2, stop_words=stop_words, max_features=5000)

print("Converting text to features...")
X = vectorizer.fit_transform(df['cleaned_text'])

# Train LDA (5 topics for 5 issue categories)
print("Training LDA model...")
lda = LatentDirichletAllocation(n_components=5, random_state=42, max_iter=10)
lda.fit(X)

print("✅ LDA training completed!\n")

# Show topics
def print_topics(model, feature_names, n_top_words=10):
    for topic_idx, topic in enumerate(model.components_):
        top_words = [feature_names[i] for i in topic.argsort()[:-n_top_words-1:-1]]
        print(f"Topic {topic_idx+1}: {', '.join(top_words)}")

print("Top words per topic:")
print_topics(lda, vectorizer.get_feature_names_out())

# Save models
os.makedirs('models', exist_ok=True)
joblib.dump(lda, 'models/lda_model.pkl')
joblib.dump(vectorizer, 'models/vectorizer.pkl')
print("\n✅ LDA model and vectorizer saved in 'models/' folder")

# Add topic features to dataset
topic_distribution = lda.transform(X)
for i in range(5):
    df[f'topic_{i+1}'] = topic_distribution[:, i]

df.to_csv('data/prepared_dataset.csv', index=False)
print("✅ Prepared dataset with LDA topics saved as 'data/prepared_dataset.csv'")

print("\n🎉 Ready for Hybrid Model Training!")

🔄 Checking for cleaned dataset...

Cleaned file not found. Loading raw dataset and cleaning...


FileNotFoundError: [Errno 2] No such file or directory: 'data/Reviews.csv'

In [3]:
# =============================================================================
# 08_LDA_Feature_Preparation.ipynb
# Hybrid Machine Learning for E-Commerce Review Issue Detection
# =============================================================================

import pandas as pd
import re
import os
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
import joblib

nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

print("🔄 Checking for cleaned dataset...\n")

# Try to load cleaned dataset, if not found → load raw and clean it
if os.path.exists('data/cleaned_dataset.csv'):
    df = pd.read_csv('data/cleaned_dataset.csv')
    print("✅ Loaded cleaned dataset")
else:
    print("Cleaned file not found. Loading raw dataset and cleaning...")
    df = pd.read_csv('data/Reviews.csv')
    df = df[['Score', 'Summary', 'Text']].copy()

    def clean_text(text):
        if not isinstance(text, str):
            return ""
        text = text.lower()
        text = re.sub(r'[^a-zA-Z\s]', '', text)
        text = re.sub(r'\s+', ' ', text).strip()
        return text

    df['cleaned_text'] = df['Text'].apply(clean_text)
    df = df[df['cleaned_text'].str.len() > 15]
    df.to_csv('data/cleaned_dataset.csv', index=False)
    print("✅ Cleaned and saved new dataset")

print(f"Total reviews: {len(df):,}\n")

# Prepare text for LDA
stop_words = stopwords.words('english')
vectorizer = CountVectorizer(max_df=0.95, min_df=2, stop_words=stop_words, max_features=5000)

print("Converting text to features...")
X = vectorizer.fit_transform(df['cleaned_text'])

# Train LDA (5 topics for 5 issue categories)
print("Training LDA model...")
lda = LatentDirichletAllocation(n_components=5, random_state=42, max_iter=10)
lda.fit(X)

print("✅ LDA training completed!\n")

# Show topics
def print_topics(model, feature_names, n_top_words=10):
    for topic_idx, topic in enumerate(model.components_):
        top_words = [feature_names[i] for i in topic.argsort()[:-n_top_words-1:-1]]
        print(f"Topic {topic_idx+1}: {', '.join(top_words)}")

print("Top words per topic:")
print_topics(lda, vectorizer.get_feature_names_out())

# Save models
os.makedirs('models', exist_ok=True)
joblib.dump(lda, 'models/lda_model.pkl')
joblib.dump(vectorizer, 'models/vectorizer.pkl')
print("\n✅ LDA model and vectorizer saved in 'models/' folder")

# Add topic features to dataset
topic_distribution = lda.transform(X)
for i in range(5):
    df[f'topic_{i+1}'] = topic_distribution[:, i]

df.to_csv('data/prepared_dataset.csv', index=False)
print("✅ Prepared dataset with LDA topics saved as 'data/prepared_dataset.csv'")

print("\n🎉 Ready for Hybrid Model Training!")

🔄 Checking for cleaned dataset...

Cleaned file not found. Loading raw dataset and cleaning...


FileNotFoundError: [Errno 2] No such file or directory: 'data/Reviews.csv'

In [4]:
# =============================================================================
# 08_LDA_Feature_Preparation.ipynb
# Final robust version - handles zip file automatically
# =============================================================================

import pandas as pd
import re
import os
import zipfile
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
import joblib

nltk.download('stopwords', quiet=True)

print("🔄 Checking data files...\n")

# Create data folder
os.makedirs('data', exist_ok=True)

# Step 1: Check if cleaned file exists
if os.path.exists('data/cleaned_dataset.csv'):
    print("✅ Found cleaned dataset. Loading it...")
    df = pd.read_csv('data/cleaned_dataset.csv')
else:
    print("No cleaned file. Checking for raw data...")

    # Check for zip file
    if os.path.exists('data/Reviews.csv.zip'):
        print("Found zip file. Extracting...")
        with zipfile.ZipFile('data/Reviews.csv.zip', 'r') as zip_ref:
            zip_ref.extractall('data/')
        print("✅ Unzipped successfully!")

    # Load raw data
    if os.path.exists('data/Reviews.csv'):
        print("Loading raw Amazon dataset...")
        df = pd.read_csv('data/Reviews.csv')
    else:
        print("❌ Raw file not found. Please run data download first.")
        raise FileNotFoundError("Reviews.csv not found")

    # Keep useful columns and clean
    df = df[['Score', 'Summary', 'Text']].copy()

    def clean_text(text):
        if not isinstance(text, str):
            return ""
        text = text.lower()
        text = re.sub(r'[^a-zA-Z\s]', '', text)
        text = re.sub(r'\s+', ' ', text).strip()
        return text

    print("Cleaning review text...")
    df['cleaned_text'] = df['Text'].apply(clean_text)
    df = df[df['cleaned_text'].str.len() > 15]

    # Save cleaned version
    df.to_csv('data/cleaned_dataset.csv', index=False)
    print("✅ Cleaned dataset saved!")

print(f"\nTotal reviews ready: {len(df):,}\n")

# --- LDA Part ---
stop_words = stopwords.words('english')
vectorizer = CountVectorizer(max_df=0.95, min_df=2, stop_words=stop_words, max_features=5000)

print("Converting text to features for LDA...")
X = vectorizer.fit_transform(df['cleaned_text'])

print("Training LDA model (5 topics)...")
lda = LatentDirichletAllocation(n_components=5, random_state=42, max_iter=10)
lda.fit(X)

print("✅ LDA training completed!\n")

# Show topics
def print_topics(model, feature_names, n_top_words=10):
    for topic_idx, topic in enumerate(model.components_):
        top_words = [feature_names[i] for i in topic.argsort()[:-n_top_words-1:-1]]
        print(f"Topic {topic_idx+1}: {', '.join(top_words)}")

print("Top words per topic:")
print_topics(lda, vectorizer.get_feature_names_out())

# Save models
os.makedirs('models', exist_ok=True)
joblib.dump(lda, 'models/lda_model.pkl')
joblib.dump(vectorizer, 'models/vectorizer.pkl')
print("\n✅ LDA model saved in 'models/' folder")

# Add topic features
topic_distribution = lda.transform(X)
for i in range(5):
    df[f'topic_{i+1}'] = topic_distribution[:, i]

df.to_csv('data/prepared_dataset.csv', index=False)
print("✅ Final prepared dataset saved as 'data/prepared_dataset.csv'")

print("\n🎉 Ready for Hybrid Model Training!")

🔄 Checking data files...

No cleaned file. Checking for raw data...
❌ Raw file not found. Please run data download first.


FileNotFoundError: Reviews.csv not found

In [5]:
# =============================================================================
# 08_LDA_Feature_Preparation.ipynb
# Hybrid Machine Learning for E-Commerce Review Issue Detection
# =============================================================================

import pandas as pd
import re
import os
import zipfile
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
import joblib

nltk.download('stopwords', quiet=True)

print("🔄 Checking and fixing data files...\n")

os.makedirs('data', exist_ok=True)

# Force extract if zip exists
if os.path.exists('data/Reviews.csv.zip'):
    print("Found zip file. Extracting now...")
    with zipfile.ZipFile('data/Reviews.csv.zip', 'r') as zip_ref:
        zip_ref.extractall('data/')
    print("✅ Unzipped successfully!")
elif os.path.exists('data/Reviews.csv'):
    print("✅ Raw CSV already exists.")
else:
    print("Downloading Amazon dataset...")
    !kaggle datasets download -d snap/amazon-fine-food-reviews -f Reviews.csv --path data/ --unzip

# Load the dataset
print("\nLoading dataset...")
df = pd.read_csv('data/Reviews.csv')

print(f"Original rows: {len(df):,}")

# Keep only useful columns
df = df[['Score', 'Summary', 'Text']].copy()

# Cleaning function
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print("Cleaning review text...")
df['cleaned_text'] = df['Text'].apply(clean_text)

# Remove very short reviews
df = df[df['cleaned_text'].str.len() > 15]

print(f"After cleaning: {len(df):,} reviews")

# Save cleaned version
df.to_csv('data/cleaned_dataset.csv', index=False)
print("✅ Cleaned dataset saved!")

# --- LDA Part ---
stop_words = stopwords.words('english')
vectorizer = CountVectorizer(max_df=0.95, min_df=2, stop_words=stop_words, max_features=5000)

print("\nConverting text to features for LDA...")
X = vectorizer.fit_transform(df['cleaned_text'])

print("Training LDA model (5 topics)...")
lda = LatentDirichletAllocation(n_components=5, random_state=42, max_iter=10)
lda.fit(X)

print("✅ LDA training completed!\n")

# Show topics
def print_topics(model, feature_names, n_top_words=10):
    for topic_idx, topic in enumerate(model.components_):
        top_words = [feature_names[i] for i in topic.argsort()[:-n_top_words-1:-1]]
        print(f"Topic {topic_idx+1}: {', '.join(top_words)}")

print("Top words per topic:")
print_topics(lda, vectorizer.get_feature_names_out())

# Save models
os.makedirs('models', exist_ok=True)
joblib.dump(lda, 'models/lda_model.pkl')
joblib.dump(vectorizer, 'models/vectorizer.pkl')
print("\n✅ LDA model and vectorizer saved!")

# Add topic features
topic_distribution = lda.transform(X)
for i in range(5):
    df[f'topic_{i+1}'] = topic_distribution[:, i]

df.to_csv('data/prepared_dataset.csv', index=False)
print("✅ Prepared dataset with LDA topics saved as 'data/prepared_dataset.csv'")

print("\n🎉 Ready for Hybrid Model Training!")

🔄 Checking and fixing data files...

Dataset URL: https://www.kaggle.com/datasets/snap/amazon-fine-food-reviews
License(s): CC0-1.0
100% 115M/115M [00:01<00:00, 86.2MB/s]


Loading dataset...


FileNotFoundError: [Errno 2] No such file or directory: 'data/Reviews.csv'

In [6]:
# =============================================================================
# 08_LDA_Feature_Preparation.ipynb
# FINAL FIXED VERSION - Handles download + unzip automatically
# =============================================================================

import pandas as pd
import re
import os
import zipfile
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
import joblib

nltk.download('stopwords', quiet=True)

print("🔄 Fixing data files and preparing for LDA...\n")

os.makedirs('data', exist_ok=True)

# Step 1: Download if raw file is missing
if not os.path.exists('data/Reviews.csv'):
    print("Downloading Amazon dataset...")
    !kaggle datasets download -d snap/amazon-fine-food-reviews -f Reviews.csv --path data/ --unzip

    # Force unzip if zip exists
    if os.path.exists('data/Reviews.csv.zip'):
        print("Extracting zip file...")
        with zipfile.ZipFile('data/Reviews.csv.zip', 'r') as zip_ref:
            zip_ref.extractall('data/')
        print("✅ Unzipped successfully!")
else:
    print("✅ Raw file already exists.")

# Step 2: Load the dataset
print("\nLoading dataset...")
df = pd.read_csv('data/Reviews.csv')

print(f"Original rows: {len(df):,}")

# Keep only useful columns
df = df[['Score', 'Summary', 'Text']].copy()

# Cleaning function
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print("Cleaning review text...")
df['cleaned_text'] = df['Text'].apply(clean_text)

# Remove very short reviews
df = df[df['cleaned_text'].str.len() > 15]

print(f"After cleaning: {len(df):,} reviews")

# Save cleaned version
df.to_csv('data/cleaned_dataset.csv', index=False)
print("✅ Cleaned dataset saved!")

# --- LDA Training ---
stop_words = stopwords.words('english')
vectorizer = CountVectorizer(max_df=0.95, min_df=2, stop_words=stop_words, max_features=5000)

print("\nConverting text to features...")
X = vectorizer.fit_transform(df['cleaned_text'])

print("Training LDA model (5 topics)...")
lda = LatentDirichletAllocation(n_components=5, random_state=42, max_iter=10)
lda.fit(X)

print("✅ LDA training completed!\n")

# Show topics
def print_topics(model, feature_names, n_top_words=10):
    for topic_idx, topic in enumerate(model.components_):
        top_words = [feature_names[i] for i in topic.argsort()[:-n_top_words-1:-1]]
        print(f"Topic {topic_idx+1}: {', '.join(top_words)}")

print("Top words per topic:")
print_topics(lda, vectorizer.get_feature_names_out())

# Save models
os.makedirs('models', exist_ok=True)
joblib.dump(lda, 'models/lda_model.pkl')
joblib.dump(vectorizer, 'models/vectorizer.pkl')
print("\n✅ LDA model saved!")

# Add topic features
topic_distribution = lda.transform(X)
for i in range(5):
    df[f'topic_{i+1}'] = topic_distribution[:, i]

df.to_csv('data/prepared_dataset.csv', index=False)
print("✅ Prepared dataset saved as 'data/prepared_dataset.csv'")

print("\n🎉 Ready for Hybrid Model Training!")

🔄 Fixing data files and preparing for LDA...

Dataset URL: https://www.kaggle.com/datasets/snap/amazon-fine-food-reviews
License(s): CC0-1.0
Reviews.csv.zip: Skipping, found more recently modified local copy (use --force to force download)
Extracting zip file...
✅ Unzipped successfully!

Loading dataset...
Original rows: 568,454
Cleaning review text...
After cleaning: 568,452 reviews
✅ Cleaned dataset saved!

Converting text to features...
Training LDA model (5 topics)...
✅ LDA training completed!

Top words per topic:
Topic 1: br, product, like, use, good, taste, water, great, oil, also
Topic 2: great, product, good, like, one, amazon, would, br, get, box
Topic 3: food, dog, br, dogs, one, cat, treats, like, cats, loves
Topic 4: coffee, like, taste, flavor, br, good, cup, drink, one, chocolate
Topic 5: tea, flavor, like, taste, good, love, great, chips, br, find

✅ LDA model saved!
✅ Prepared dataset saved as 'data/prepared_dataset.csv'

🎉 Ready for Hybrid Model Training!
